# Exploratory Data Analysis — International Football Results
This notebook is for manual exploration only (no modeling). The goal is to
build intuition for the data before any feature engineering happens, so you
can sanity-check what the pipeline produces later against what you see here.

Place this notebook in `notebooks/exploratory/` and run it from there
(paths below assume `../../data/raw/`). Adjust the path if you run it
elsewhere.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

results = pd.read_csv('../../data/raw/results.csv', parse_dates=['date'])
goalscorers = pd.read_csv('../../data/raw/goalscorers.csv', parse_dates=['date'])
shootouts = pd.read_csv('../../data/raw/shootouts.csv', parse_dates=['date'])
former_names = pd.read_csv('../../data/raw/former_names.csv', parse_dates=['start_date', 'end_date'])

print(results.shape, goalscorers.shape, shootouts.shape, former_names.shape)

## 1. results.csv — basic shape and quality

In [ ]:
print("Date range:", results['date'].min(), "to", results['date'].max())
print("\nNulls:\n", results.isnull().sum())
print("\nRows with null scores (future/unplayed fixtures):")
results[results['home_score'].isnull() | results['away_score'].isnull()]

In [ ]:
# duplicate check on the key that matters for chronological processing
dupes = results[results.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)]
print("Duplicate (date, home_team, away_team) rows:", len(dupes))
dupes.sort_values(['date','home_team']).head(10)

## 2. Team and tournament coverage

In [ ]:
all_teams = set(results['home_team']) | set(results['away_team'])
print("Unique team names:", len(all_teams))

tournament_counts = results['tournament'].value_counts()
print("\nNumber of distinct tournament labels:", len(tournament_counts))
tournament_counts.head(20)

In [ ]:
tournament_counts.head(15).plot(kind='barh', figsize=(8,6))
plt.title('Top 15 tournament types by match count')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Home advantage — does home_team actually win more?

In [ ]:
def outcome(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    return 'draw'

results_clean = results.dropna(subset=['home_score', 'away_score']).copy()
results_clean['outcome'] = results_clean.apply(outcome, axis=1)

print("Overall outcome distribution:")
print(results_clean['outcome'].value_counts(normalize=True))

print("\nOutcome distribution when NOT neutral venue:")
print(results_clean[results_clean['neutral'] == False]['outcome'].value_counts(normalize=True))

print("\nOutcome distribution when neutral venue (home advantage should shrink):")
print(results_clean[results_clean['neutral'] == True]['outcome'].value_counts(normalize=True))

**What to look for:** home win rate should be noticeably higher than away win
rate in non-neutral matches, and that gap should shrink (though maybe not
vanish — "true neutral" is debatable, e.g. regional tournaments) in neutral
matches. If home advantage doesn't show up at all here, that's a red flag
worth understanding before you build Elo with a home_advantage constant.

## 4. Goals over time — has the game gotten higher or lower scoring?

In [ ]:
results_clean['year'] = results_clean['date'].dt.year
results_clean['total_goals'] = results_clean['home_score'] + results_clean['away_score']

avg_goals_per_year = results_clean.groupby('year')['total_goals'].mean()
avg_goals_per_year.plot(figsize=(12,5))
plt.title('Average total goals per match, by year')
plt.ylabel('Avg goals/match')
plt.show()

## 5. Draw rate over time — is 'draw' getting harder or easier to predict across eras?

In [ ]:
draw_rate_by_year = results_clean.groupby('year')['outcome'].apply(lambda x: (x=='draw').mean())
draw_rate_by_year.plot(figsize=(12,5))
plt.title('Draw rate by year')
plt.ylabel('Proportion of matches drawn')
plt.show()

## 6. Score margin distribution — sanity check for the Poisson assumption

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
results_clean['home_score'].clip(upper=8).value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Home score distribution (clipped at 8)')
results_clean['away_score'].clip(upper=8).value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Away score distribution (clipped at 8)')
plt.tight_layout()
plt.show()

print("Home goals mean/var:", results_clean['home_score'].mean(), results_clean['home_score'].var())
print("Away goals mean/var:", results_clean['away_score'].mean(), results_clean['away_score'].var())

**What to look for:** for a Poisson model to be a reasonable fit, mean and
variance of goals should be roughly similar (Poisson assumes mean = variance).
If variance is much higher than the mean, that's "overdispersion" and you may
need a Negative Binomial model instead down the line — worth knowing now,
not discovering after training.

## 7. goalscorers.csv — own goals and penalties

In [ ]:
print("Own goal rate:", goalscorers['own_goal'].mean())
print("Penalty rate:", goalscorers['penalty'].mean())
print("\nNull scorer rows (data quality):", goalscorers['scorer'].isnull().sum())
print("Null minute rows:", goalscorers['minute'].isnull().sum())

## 8. shootouts.csv — does shooting first actually help?

In [ ]:
so = shootouts.dropna(subset=['first_shooter']).copy()
so['first_shooter_won'] = so['first_shooter'] == so['winner']
print("Shootouts with known first shooter:", len(so))
print("Win rate when shooting first:", so['first_shooter_won'].mean())

**What to look for:** this is a small sample (fewer than 681 rows, and many
missing `first_shooter`), so don't over-interpret a small edge either way —
this is a "curiosity check," not something to build a model on with
confidence.

## 9. former_names.csv — team identity changes

In [ ]:
former_names.sort_values('start_date')

**What to look for:** confirm none of these former names still appear as
`home_team`/`away_team` values in `results.csv` — the dataset documentation
says current names are already used throughout, so this should come back
empty.

In [ ]:
former_name_set = set(former_names['former'])
leaked_old_names = all_teams & former_name_set
print("Former names still appearing in results.csv (should be empty):", leaked_old_names)

## 10. Match frequency over time — how has global participation grown?

In [ ]:
matches_per_year = results_clean.groupby('year').size()
matches_per_year.plot(figsize=(12,5))
plt.title('Number of international matches per year')
plt.ylabel('Match count')
plt.show()

teams_active_per_year = results_clean.groupby('year').apply(
    lambda d: len(set(d['home_team']) | set(d['away_team']))
)
teams_active_per_year.plot(figsize=(12,5))
plt.title('Number of distinct teams active per year')
plt.ylabel('Active teams')
plt.show()